# Experimento 06 — Conversão e validação Keras → TensorFlow Lite

Objetivos deste notebook:

1. carregar o melhor modelo do Experimento 06;
2. converter o modelo Keras para TensorFlow Lite em `float32`, sem quantização;
3. validar numericamente a equivalência entre Keras e TFLite;
4. salvar entradas e saídas de referência para o benchmark na Raspberry Pi Zero 2 W;
5. comparar o tamanho dos artefatos em disco.

> O modelo Keras original permanece inalterado.


In [1]:
import os
import time
import numpy as np
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)


TensorFlow: 2.21.0
NumPy: 2.5.1


## 1. Configuração dos arquivos

Coloque este notebook na mesma pasta do arquivo `melhor_modelo_exp06.keras`.
Se o seu arquivo estiver com outro nome, altere apenas `KERAS_MODEL_PATH`.


In [2]:
KERAS_MODEL_PATH = "melhor_modelo_exp06.keras"
TFLITE_MODEL_PATH = "melhor_modelo_exp06.tflite"

BENCHMARK_INPUT_PATH = "benchmark_input.npy"
KERAS_OUTPUT_PATH = "benchmark_output_keras.npy"
TFLITE_OUTPUT_PATH = "benchmark_output_tflite.npy"

ALTURA = 128
LARGURA = 128
CANAIS = 1
SEMENTE = 0

assert os.path.exists(KERAS_MODEL_PATH), (
    f"Arquivo não encontrado: {KERAS_MODEL_PATH}"
)

print("Modelo Keras:", KERAS_MODEL_PATH)


Modelo Keras: melhor_modelo_exp06.keras


## 2. Carregamento do modelo

Usamos `compile=False` porque a função de perda personalizada (`mae_ssim_loss`) foi necessária
durante o treinamento, mas **não é necessária para inferência nem para conversão TFLite**.
Isso também evita a necessidade de redefinir a loss neste notebook.


In [3]:
model = tf.keras.models.load_model(
    KERAS_MODEL_PATH,
    compile=False
)

print("Modelo carregado com sucesso.")
print("Entrada:", model.input_shape)
print("Saída:", model.output_shape)

model.summary()


Modelo carregado com sucesso.
Entrada: (None, None, None, 1)
Saída: (None, None, None, 1)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ None, 1)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, None,      │        640 │ input_layer[0][0] │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, None,      │     36,928 │ conv2d[0][0]      │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, None,      │     36,928 │ conv2d_1[0][0]    │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, None,      │     36,928 │ conv2d_2[0][0]    │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, None,      │     36,928 │ conv2d_3[0][0]    │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, None,      │     36,928 │ conv2d_4[0][0]    │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ predicted_noise     │ (None, None,      │        577 │ conv2d_5[0][0]    │
│ (Conv2D)            │ None, 1)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ restored_image      │ (None, None,      │          0 │ input_layer[0][0… │
│ (Subtract)          │ None, 1)          │            │ predicted_noise[… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 185,857 (726.00 KB)

 Trainable params: 185,857 (726.00 KB)

 Non-trainable params: 0 (0.00 B)

## 3. Conversão para TensorFlow Lite FP32

Nesta primeira implantação não aplicamos quantização. Assim, isolamos a mudança de runtime:

`Keras FP32 → TFLite FP32`


In [4]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Sem quantização nesta etapa.
converter.optimizations = []

tflite_model = converter.convert()

with open(TFLITE_MODEL_PATH, "wb") as f:
    f.write(tflite_model)

print(
    f"Modelo TFLite salvo em: {TFLITE_MODEL_PATH}"
)


INFO:tensorflow:Assets written to: C:\Users\cayoc\AppData\Local\Temp\tmpfld3uuzx\assets


INFO:tensorflow:Assets written to: C:\Users\cayoc\AppData\Local\Temp\tmpfld3uuzx\assets


Saved artifact at 'C:\Users\cayoc\AppData\Local\Temp\tmpfld3uuzx'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, None, None, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, None, None, 1), dtype=tf.float32, name=None)
Captures:
  2244500717648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2244500718992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2244500718416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2244500719568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2244500719376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2244500718608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2244500719760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2244500720336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2244500720144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2244500720720: TensorSpec(shape=(), dtype=tf.resource, name=

## 4. Preparação de uma entrada de referência

Para validar a conversão e posteriormente conferir a execução na Raspberry Pi,
criamos uma entrada `float32` determinística no formato utilizado pelo treinamento
com patches: `(1, 128, 128, 1)`.

Essa entrada é destinada à **validação numérica e benchmark de inferência**, não à
avaliação científica de qualidade do denoising.


In [5]:
rng = np.random.default_rng(SEMENTE)

benchmark_input = rng.random(
    (1, ALTURA, LARGURA, CANAIS),
    dtype=np.float32
)

np.save(
    BENCHMARK_INPUT_PATH,
    benchmark_input
)

print("Entrada:", benchmark_input.shape)
print("dtype:", benchmark_input.dtype)
print("min:", benchmark_input.min())
print("max:", benchmark_input.max())


Entrada: (1, 128, 128, 1)
dtype: float32
min: 1.335144e-05
max: 0.9999967


## 5. Inferência com o modelo Keras


In [6]:
inicio = time.perf_counter()

keras_output = model(
    benchmark_input,
    training=False
).numpy()

tempo_keras = time.perf_counter() - inicio

np.save(
    KERAS_OUTPUT_PATH,
    keras_output
)

print("Saída Keras:", keras_output.shape)
print("Tempo Keras:", tempo_keras, "s")


Saída Keras: (1, 128, 128, 1)
Tempo Keras: 0.4582438000000195 s


## 6. Inferência com TensorFlow Lite

O modelo foi criado com dimensões espaciais variáveis. Por isso, antes de alocar os tensores,
redimensionamos explicitamente a entrada do interpretador para `(1, 128, 128, 1)`.


In [7]:
interpreter = tf.lite.Interpreter(
    model_path=TFLITE_MODEL_PATH
)

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Entrada TFLite antes do resize:")
print(input_details[0])

interpreter.resize_tensor_input(
    input_details[0]["index"],
    benchmark_input.shape,
    strict=False
)

interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("\nEntrada TFLite após o resize:")
print(input_details[0])

print("\nSaída TFLite:")
print(output_details[0])


Entrada TFLite antes do resize:
{'name': 'serving_default_input_layer:0', 'index': 0, 'shape': array([1, 1, 1, 1], dtype=int32), 'shape_signature': array([-1, -1, -1,  1], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}, 'sparsity_parameters': {}}

Entrada TFLite após o resize:
{'name': 'serving_default_input_layer:0', 'index': 0, 'shape': array([  1, 128, 128,   1], dtype=int32), 'shape_signature': array([-1, -1, -1,  1], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}, 'sparsity_parameters': {}}

Saída TFLite:
{'name': 'StatefulPartitionedCall_1:0', 'index': 22, 'shape': array([  1, 128, 128,   1], dtype=int32), 'shape_signature': array([-1, -1, 

c:\Doutorado\termografia\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [8]:
interpreter.set_tensor(
    input_details[0]["index"],
    benchmark_input
)

inicio = time.perf_counter()

interpreter.invoke()

tempo_tflite = time.perf_counter() - inicio

tflite_output = interpreter.get_tensor(
    output_details[0]["index"]
)

np.save(
    TFLITE_OUTPUT_PATH,
    tflite_output
)

print("Saída TFLite:", tflite_output.shape)
print("Tempo TFLite:", tempo_tflite, "s")


Saída TFLite: (1, 128, 128, 1)
Tempo TFLite: 0.23692330000000084 s


## 7. Validação numérica Keras × TFLite

A conversão é considerada válida quando as diferenças introduzidas pelo runtime TFLite são
numericamente desprezíveis para a mesma entrada.


In [9]:
diferenca = keras_output - tflite_output
abs_diferenca = np.abs(diferenca)

max_abs_diff = float(np.max(abs_diferenca))
mean_abs_diff = float(np.mean(abs_diferenca))
rmse_diff = float(
    np.sqrt(
        np.mean(
            diferenca ** 2
        )
    )
)

equivalente_1e5 = np.allclose(
    keras_output,
    tflite_output,
    rtol=1e-5,
    atol=1e-5
)

equivalente_1e6 = np.allclose(
    keras_output,
    tflite_output,
    rtol=1e-6,
    atol=1e-6
)

print("--- Diferença Keras × TFLite ---")
print(f"Máxima diferença absoluta : {max_abs_diff:.10e}")
print(f"Média diferença absoluta  : {mean_abs_diff:.10e}")
print(f"RMSE entre as saídas      : {rmse_diff:.10e}")
print()
print("allclose (1e-5):", equivalente_1e5)
print("allclose (1e-6):", equivalente_1e6)


--- Diferença Keras × TFLite ---
Máxima diferença absoluta : 2.3841857910e-07
Média diferença absoluta  : 2.5780209967e-08
RMSE entre as saídas      : 4.0343895336e-08

allclose (1e-5): True
allclose (1e-6): True


## 8. Validação em múltiplas entradas

Uma única entrada pode ocultar algum caso particular. Portanto, repetimos a comparação
com várias entradas determinísticas.


In [10]:
def inferir_tflite(interpreter, input_details, output_details, entrada):
    interpreter.resize_tensor_input(
        input_details[0]["index"],
        entrada.shape,
        strict=False
    )

    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(
        input_details[0]["index"],
        entrada.astype(np.float32)
    )

    interpreter.invoke()

    saida = interpreter.get_tensor(
        output_details[0]["index"]
    )

    return saida, input_details, output_details


rng_validacao = np.random.default_rng(SEMENTE + 1)

max_diffs = []
mean_diffs = []

for i in range(10):

    entrada = rng_validacao.random(
        (1, ALTURA, LARGURA, CANAIS),
        dtype=np.float32
    )

    saida_keras = model(
        entrada,
        training=False
    ).numpy()

    saida_tflite, input_details, output_details = inferir_tflite(
        interpreter,
        input_details,
        output_details,
        entrada
    )

    diff = np.abs(
        saida_keras - saida_tflite
    )

    max_diffs.append(
        float(np.max(diff))
    )

    mean_diffs.append(
        float(np.mean(diff))
    )

print("Validação com 10 entradas:")
print(
    "Maior diferença absoluta observada:",
    f"{max(max_diffs):.10e}"
)

print(
    "Média das diferenças médias:",
    f"{np.mean(mean_diffs):.10e}"
)


Validação com 10 entradas:
Maior diferença absoluta observada: 3.5762786865e-07
Média das diferenças médias: 2.5666540182e-08


## 9. Tamanho dos modelos


In [11]:
keras_size = os.path.getsize(
    KERAS_MODEL_PATH
)

tflite_size = os.path.getsize(
    TFLITE_MODEL_PATH
)

print(
    f"Keras : {keras_size / (1024 ** 2):.3f} MiB"
)

print(
    f"TFLite: {tflite_size / (1024 ** 2):.3f} MiB"
)

print(
    f"Razão TFLite/Keras: {tflite_size / keras_size:.3f}"
)


Keras : 2.178 MiB
TFLite: 0.714 MiB
Razão TFLite/Keras: 0.328


## 10. Arquivos gerados para a Raspberry Pi

Após a validação, estes arquivos serão utilizados no benchmark embarcado:

- `melhor_modelo_exp06.tflite`
- `benchmark_input.npy`
- `benchmark_output_keras.npy`

Na Raspberry Pi Zero 2 W, a mesma `benchmark_input.npy` será processada pelo runtime TFLite.
A saída obtida será comparada com `benchmark_output_keras.npy`, além da medição de latência,
memória, temperatura e throttling.


In [12]:
arquivos = [
    TFLITE_MODEL_PATH,
    BENCHMARK_INPUT_PATH,
    KERAS_OUTPUT_PATH,
    TFLITE_OUTPUT_PATH
]

print("Arquivos preparados:")

for arquivo in arquivos:
    tamanho_kib = os.path.getsize(
        arquivo
    ) / 1024

    print(
        f"- {arquivo}: {tamanho_kib:.2f} KiB"
    )


Arquivos preparados:
- melhor_modelo_exp06.tflite: 730.94 KiB
- benchmark_input.npy: 64.12 KiB
- benchmark_output_keras.npy: 64.12 KiB
- benchmark_output_tflite.npy: 64.12 KiB
